<a id="inizio"></a>
# 12 · Capstone: il carico del Nord e la temperatura

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Avanzata &nbsp;·&nbsp; ⏱ ~130 min**

Due anni di carico elettrico della zona Nord e la temperatura di Milano: dalla domanda del capo al grafico che risponde, passando per tutto quello che abbiamo imparato.

**In questo notebook impariamo a**

- caricare, pulire e aggregare una serie temporale reale, con i suoi difetti
- unire due fonti diverse e leggere la relazione tra carico e temperatura
- chiudere un'analisi con numeri, frasi e un grafico da mandare a chi non ha Python

Prima di questo: [11 · Agenti per il coding](11_Agenti_per_il_coding.ipynb) &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/load_total_north_hourly_2025.xlsx`, `../Dati/fallback/meteo_milano_2024_2025.json`

</div>

<a id="indice"></a>
**Indice**

- [1. I dati di Terna](#sez-1)
    - [Step 1 · Caricare e mettere in ordine](#es-1)
- [2. L'ora legale](#sez-2)
    - [Step 2 · Pulizia temporale](#es-2)
- [3. Un anno in un grafico](#sez-3)
    - [Step 3 · La serie giornaliera](#es-3)
- [4. La giornata tipo](#sez-4)
    - [Step 4 · I profili](#es-4)
- [5. La temperatura di Milano](#sez-5)
    - [Step 5 · Scaricare la temperatura](#es-5)
- [6. Due tabelle, una sola](#sez-6)
    - [Step 6 · L'unione](#es-6)
- [7. La relazione](#sez-7)
    - [Step 7 · Carico e temperatura](#es-7)
- [8. Il report](#sez-8)
    - [Step 8 · La sintesi](#es-8)

Lunedì mattina, riunione di pianificazione. Il capo: "Il carico del Nord dipende dalla temperatura?
Quanto cala nel weekend? E l'estate pesa più dell'inverno, da noi?". Abbiamo i file di Terna con due
anni di carico della zona Nord e un'API meteo. Niente previsioni: prima bisogna capire i dati.

Otto step, uno per sezione. Ogni step inizia con un'indicazione su Copilot: *da soli* dove conta
capire il meccanismo, *per il dettaglio* dove può sistemare titoli ed etichette, *chat per capire*
quando qualcosa non torna. Le verifiche controllano forma e intervalli: il resto lo controlli tu,
guardando l'output.

Cose da scoprire lungo la strada, non da leggere qui: la frequenza vera dei dati, quanti picchi ha
una giornata, di quanto cala il weekend, se al Nord pesa più il caldo o il freddo, che forma ha la
relazione tra carico e temperatura, e cosa combina l'ora legale due volte l'anno.

<a id="sez-1"></a>
## 1. I dati di Terna

*Copilot: da soli.*

Due file Excel, uno per anno, con la stessa struttura: `Date`, `Total Load [MW]`,
`Forecast Total Load [MW]`, `Bidding Zone`. Ci serve una tabella sola, con due colonne dai nomi
corti e le righe in ordine di tempo. Il forecast di Terna lo lasciamo a Terna.

<a id="es-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 1 · Caricare e mettere in ordine**

I file sono nella cartella `../Dati/`. Guarda le prime righe di uno dei due prima di partire: c'è una sorpresa sull'ordine.

**Cosa fare.**

Costruisci `carico`: i due anni in una tabella sola, due colonne `data` (datetime) e `mw`,
righe in ordine cronologico con indice da 0, senza il forecast e la zona.
Risultato atteso: `carico.info()` con 70176 righe e due colonne.

</div>

In [ ]:
import pandas as pd
import plotly.express as px

carico = ...
carico.info()

In [ ]:
# Verifica: esegui senza modificare
assert set(carico.columns) == {"data", "mw"}, "❌ carico: due colonne sole, data e mw"
assert len(carico) == 70176, "❌ carico: 70176 righe, i due anni uno sotto l'altro senza togliere nulla per ora"
assert str(carico["data"].dtype).startswith("datetime64"), "❌ La colonna data deve essere datetime"
assert carico["data"].is_monotonic_increasing, "❌ Ordina per data: i file arrivano al contrario"
print("✅ Step 1 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`pd.concat([a, b], ignore_index=True)` mette le righe una sotto l'altra; `drop(columns=...)`
e `rename(columns={...})` sistemano le colonne; `sort_values` e `reset_index(drop=True)`
l'ordine. Un metodo per riga: si legge meglio e si sbaglia meno.

</div>

<a id="sez-2"></a>
## 2. L'ora legale

*Copilot: da soli.*

Il file si chiama `hourly`. Prima di fidarci del nome, misuriamo il passo tra un timestamp e il
successivo: lo abbiamo già fatto sui prezzi zonali.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

In `passi` metti la differenza tra ogni timestamp e il precedente, in `frequenza` quante volte
compare ogni valore. Qual è il passo vero? E cosa sono gli altri due valori?

</div>

In [ ]:
passi = carico["data"].diff()
frequenza = ...
frequenza

In [ ]:
# Verifica: esegui senza modificare
assert frequenza.idxmax() == pd.Timedelta("15min"), "❌ Il passo più frequente deve essere di 15 minuti: value_counts su diff"
assert frequenza[pd.Timedelta(0)] == 8, "❌ Dovrebbero esserci 8 passi di zero minuti: timestamp ripetuti"
assert frequenza[pd.Timedelta("1h15min")] == 2, "❌ Dovrebbero esserci 2 passi di un'ora e un quarto: buchi"
print("✅ Tutto corretto")

Quartorario, non orario: 96 valori al giorno. Otto passi di zero minuti sono timestamp ripetuti;
due passi di un'ora e un quarto sono buchi di quattro quartorari. Vediamo dove stanno.

<a id="es-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 2 · Pulizia temporale**

Due anni, otto doppioni e otto quartorari che mancano: sempre negli stessi due giorni dell'anno. Prima di togliere qualcosa, guardiamo che giorni sono.

**Cosa fare.**

Trova i timestamp ripetuti in `duplicati` (tutte le copie) e, in `giorni_strani`, i giorni
che non hanno 96 quartorari (`quartorari_giorno` è il conteggio per giorno). Poi lascia in
`carico` una riga per timestamp, indice da 0, e conta i valori mancanti.
Risultato atteso: 8 righe doppie, 4 giorni strani, 70168 righe pulite, zero NaN.
Domanda: perché quei quattro giorni, e perché il buco di marzo non va riempito?

</div>

In [ ]:
duplicati = ...
quartorari_giorno = ...
giorni_strani = ...

carico = ...
carico.isna().sum()

In [ ]:
# Verifica: esegui senza modificare
assert len(duplicati) == 16, "❌ duplicati: 16 righe, cioè 4 timestamp doppi per anno in 2 copie (keep=False le tiene tutte)"
assert set(pd.to_datetime(list(giorni_strani.index)).strftime("%Y-%m-%d")) == {"2024-03-31", "2024-10-27", "2025-03-30", "2025-10-26"}, "❌ giorni_strani: le ultime domeniche di marzo e di ottobre dei due anni"
assert len(carico) == 70168, "❌ carico: 70176 righe meno gli 8 duplicati"
assert carico["data"].duplicated().sum() == 0, "❌ Ci sono ancora timestamp ripetuti"
assert carico["mw"].isna().sum() == 0, "❌ Non dovrebbero esserci valori mancanti"
print("✅ Step 2 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`duplicated(keep=False)` segna tutte le copie, `drop_duplicates(subset="data")` tiene la
prima. Per contare per giorno: `groupby(carico["data"].dt.date).size()`.

</div>

Perché proprio quelle date? Chiediamolo a pandas. Questa cella dà errore apposta: leggiamolo
dal basso.

In [ ]:
carico["data"].dt.tz_localize("Europe/Rome", ambiguous="infer")

`AmbiguousTimeError`: le 2:00 dell'ultima domenica di ottobre esistono due volte nel fuso
orario italiano, e con una copia sola rimasta pandas non sa più quale delle due sia. Per questa
analisi l'ora locale senza fuso basta: il rimedio è stato togliere i doppioni.

<a id="sez-3"></a>
## 3. Un anno in un grafico

*Copilot: da soli.*

Settantamila punti non si guardano: li raggruppiamo per giorno. `resample("D")` lavora su un
indice temporale e raggruppa per giorno di calendario, come un `groupby` che sa che i giorni sono
consecutivi; il risultato ha la data come indice e `reset_index()` la riporta colonna.

<a id="es-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 3 · La serie giornaliera**

Il capo vuole "vedere i due anni". Un grafico con lo slider sotto, e due numeri: il giorno più leggero e quello più pesante.

**Cosa fare.**

Costruisci `giornaliero`: una riga per giorno, colonne `data` e `mwh` (energia del giorno:
un quartorario a P MW vale P/4 MWh). Disegnala con lo slider sotto e metti in `estremi` le
due righe del giorno più leggero e di quello più pesante.
Risultato atteso: una tabella 731×2, un grafico a una linea, una tabella 2×2.
Domanda: cosa succede ad agosto e nella settimana di Natale?

</div>

In [ ]:
giornaliero = ...

estremi = ...
estremi

In [ ]:
# Verifica: esegui senza modificare
assert set(giornaliero.columns) == {"data", "mwh"}, "❌ giornaliero: due colonne, data e mwh"
assert len(giornaliero) == 731, "❌ giornaliero: 366 + 365 giorni"
assert giornaliero["mwh"].between(250_000, 700_000).all(), "❌ mwh fuori scala: hai diviso per 4 i MW quartorari?"
assert len(estremi) == 2, "❌ estremi: due righe, il giorno più leggero e quello più pesante"
assert estremi["mwh"].min() < 300_000 and estremi["mwh"].max() > 640_000, "❌ estremi: il minimo sta sotto i 300.000 MWh, il massimo sopra i 640.000"
print("✅ Step 3 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`set_index("data")`, poi `["mw"].resample("D").sum()`: il risultato ha la data come indice,
`reset_index()` la riporta colonna. Lo slider: `fig.update_xaxes(rangeslider_visible=True)`.

</div>

<a id="sez-4"></a>
## 4. La giornata tipo

*Copilot: per il dettaglio (titoli, etichette, colori), non per la logica.*

Un'altra domanda sugli stessi dati: com'è fatta una giornata media? Non serve il tempo come
indice, basta l'ora del giorno in una colonna e un `groupby`.

<a id="es-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 4 · I profili**

Il collega della sala controllo giura che i picchi sono due. Il commerciale vuole sapere quanto cala il weekend, in percentuale, per un'offerta.

**Cosa fare.**

Aggiungi a `carico` le colonne `ora` e `weekend` (True il sabato e la domenica). Costruisci
`profilo` (carico medio per ora: tabella 24×2) e `profilo_tipo` (lo stesso, separato per
`weekend`: 48 righe) e disegna `profilo_tipo` a due linee. Calcola `calo_weekend`: di quanto
il carico medio del weekend sta sotto quello feriale, in percentuale con un decimale.
Domande: quanti picchi, a che ore? A che ora il minimo?

</div>

In [ ]:
carico["ora"] = ...
carico["weekend"] = ...

profilo = ...
profilo_tipo = ...
calo_weekend = ...
print(f"Calo del weekend: {calo_weekend}%")
profilo

In [ ]:
# Verifica: esegui senza modificare
assert profilo.shape == (24, 2), "❌ profilo: 24 righe (le ore) e 2 colonne (ora e mw): serve reset_index"
assert profilo["ora"].nunique() == 24, "❌ profilo: manca qualche ora"
assert 3 <= profilo.loc[profilo["mw"].idxmin(), "ora"] <= 6, "❌ Il minimo del profilo dovrebbe cadere di notte, tra le 3 e le 6"
assert len(profilo_tipo) == 48, "❌ profilo_tipo: 24 ore per 2 tipi di giorno = 48 righe"
assert 20 <= calo_weekend <= 30, "❌ calo_weekend: tra 20 e 30 per cento, con un decimale"
print("✅ Step 4 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`dt.hour` e `dt.dayofweek >= 5` fanno le due colonne; `groupby(["weekend", "ora"])` dà una
riga per combinazione. Per il calo: medie per `weekend`, poi `1 - weekend / feriale`.

</div>

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Le due linee dicono poco sul sabato rispetto alla domenica. Costruisci `mappa`: una griglia
con i giorni della settimana sulle righe (0 = lunedì) e le ore sulle colonne, carico medio
dentro; poi colorala con `px.imshow(mappa, aspect="auto")`. Dove sta la cella più chiara?

</div>

In [ ]:
carico["giorno_settimana"] = carico["data"].dt.dayofweek
mappa = carico.pivot_table(index=..., columns=..., values=..., aggfunc="mean")

fig = px.imshow(mappa, aspect="auto", labels={"x": "ora", "y": "giorno (0 = lunedì)", "color": "MW"},
                title="Carico medio per giorno della settimana e ora")
fig.show()

In [ ]:
# Verifica: esegui senza modificare
assert mappa.shape == (7, 24), "❌ mappa: 7 giorni sulle righe, 24 ore sulle colonne"
assert mappa.loc[6].mean() < mappa.loc[2].mean(), "❌ La domenica (6) dovrebbe stare sotto il mercoledì (2)"
print("✅ Tutto corretto")

<a id="sez-5"></a>
## 5. La temperatura di Milano

*Copilot: per il dettaglio (titoli, etichette, colori), non per la logica.*

La temperatura arriva dall'archivio storico di Open-Meteo: oraria, per un punto. Prendiamo Milano
come rappresentante del Nord: non lo è del tutto, ma per capire la forma della relazione basta.

<a id="es-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 5 · Scaricare la temperatura**

Due anni di temperatura oraria, dall'archivio storico di Open-Meteo. La chiamata la scriviamo noi, sullo schema della previsione.

**Cosa fare.**

Chiedi all'archivio storico di Open-Meteo la temperatura oraria di Milano (latitudine
45.4642, longitudine 9.19) dal primo all'ultimo giorno del carico. URL
`https://archive-api.open-meteo.com/v1/archive`; parametri `latitude`, `longitude`,
`start_date` ed `end_date` come testo `AAAA-MM-GG`, `hourly="temperature_2m"`,
`timezone="Europe/Rome"`. Trasforma la risposta in `meteo` con due colonne: `data` (datetime)
e `temperatura`. Disegnala e chiudi con `describe()`.
Risultato atteso: 17544 righe, temperature plausibili per Milano.

</div>

In [ ]:
import requests

meteo = ...
meteo.describe()

In [ ]:
# Verifica: esegui senza modificare
assert set(meteo.columns) == {"data", "temperatura"}, "❌ meteo: due colonne, data e temperatura"
assert str(meteo["data"].dtype).startswith("datetime64"), "❌ meteo: la colonna data deve essere datetime"
assert len(meteo) == 17544, "❌ meteo: un valore per ogni ora dei due anni, 17544"
assert meteo["temperatura"].between(-15, 45).all(), "❌ meteo: temperature fuori da ogni plausibilità per Milano"
print("✅ Step 5 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`requests.get(url, params=params, timeout=30)`, poi `raise_for_status()` e `.json()`. Nella
risposta, `["hourly"]` è un dizionario con due liste, `time` e `temperature_2m`:
`pd.DataFrame(...)` lo trasforma in tabella, poi `to_datetime` su `time` e `rename`.

</div>

Se l'API non risponde, esegui la cella qui sotto al posto di quella sopra: produce lo stesso
`meteo` dalla copia salvata. Poi rilancia la verifica.

In [ ]:
import json

with open("../Dati/fallback/meteo_milano_2024_2025.json", encoding="utf-8") as f:
    risposta = json.load(f)

meteo = pd.DataFrame(risposta["hourly"])
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.rename(columns={"time": "data", "temperature_2m": "temperatura"})
meteo.describe()

<a id="sez-6"></a>
## 6. Due tabelle, una sola

*Copilot: per il dettaglio (titoli, etichette, colori), non per la logica.*

La temperatura è oraria, il carico quartorario: prima di unirli, portiamo il carico all'ora.
Questa parte usa solo il carico, quindi funziona anche senza rete.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Costruisci `carico_orario`: media dei quattro quartorari di ogni ora, con `data` e `mw` come
colonne, e mostra le righe con valori mancanti. Risultato atteso: 17544 righe. Domanda: da dove
vengono i NaN?

</div>

In [ ]:
carico_orario = ...

print(len(carico_orario), "ore")
carico_orario[carico_orario["mw"].isna()]

In [ ]:
# Verifica: esegui senza modificare
assert set(carico_orario.columns) == {"data", "mw"}, "❌ carico_orario: due colonne, data e mw"
assert len(carico_orario) == 17544, "❌ carico_orario: 17544 ore; con resample compare anche l'ora di marzo che manca, vuota"
print("✅ Tutto corretto")

Le due ore vuote sono le 2:00 dell'ultima domenica di marzo: `resample` costruisce tutte le ore
del calendario, anche quella che nei dati non c'è. Ora il `merge`: a ogni ora del carico la sua
temperatura, come un CERCA.VERT.

<a id="es-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 6 · L'unione**

Due tabelle con la stessa colonna `data`. Prima e dopo il merge contiamo le righe: è il controllo che smaschera una chiave non unica.

**Cosa fare.**

Unisci `carico_orario` e `meteo` in `unione` (colonne `data`, `mw`, `temperatura`) tenendo
tutte le ore del carico. Stampa le righe prima e dopo, conta i valori mancanti per colonna,
poi togli le righe incomplete.
Risultato atteso: circa 17540 righe, zero NaN. Domanda: da dove venivano i NaN?

</div>

In [ ]:
unione = ...
unione.head()

In [ ]:
# Verifica: esegui senza modificare
assert set(unione.columns) == {"data", "mw", "temperatura"}, "❌ unione: tre colonne, data, mw e temperatura"
assert unione.isna().sum().sum() == 0, "❌ unione: dopo dropna non devono restare valori mancanti"
assert 17000 <= len(unione) <= 17548, "❌ unione: circa 17500 righe, una per ora"
print("✅ Step 6 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`pd.merge(carico_orario, meteo, on="data", how="left")`; `len()` prima e dopo dice se la
chiave era unica; `isna().sum()` conta i buchi per colonna; `dropna()` li toglie, riassegnando.

</div>

<a id="sez-7"></a>
## 7. La relazione

*Copilot: per il dettaglio (titoli, etichette, colori), non per la logica.*

Ora la domanda del capo. Un punto per giorno: temperatura media sulle x, carico medio sulle y, un
colore per mese. La forma che esce è la risposta.

<a id="es-7"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 7 · Carico e temperatura**

Il capo si aspetta una retta. Vediamo cosa dicono 731 giorni.

**Cosa fare.**

Porta `unione` a medie giornaliere in `giornaliero_tc` (colonne `data`, `mw`,
`temperatura`, più `mese`) e disegna lo scatter temperatura contro carico, un colore per
mese. Poi trova `temperatura_minimo`: il bordo inferiore della fascia di 3 °C in cui il
carico medio è più basso.
Risultato atteso: 731 punti con una forma riconoscibile, e una fascia.
Domande: pesa di più il freddo o il caldo? Dove stanno i punti bassi di ogni colore?

</div>

In [ ]:
giornaliero_tc = ...

temperatura_minimo = ...
print(f"Carico minimo nella fascia {temperatura_minimo:.0f}-{temperatura_minimo + 3:.0f} °C")

In [ ]:
# Verifica: esegui senza modificare
assert len(giornaliero_tc) == 731, "❌ giornaliero_tc: una riga per giorno, 731"
assert giornaliero_tc["mese"].nunique() == 12, "❌ mese: i dodici mesi, da dt.month"
assert 10 <= temperatura_minimo <= 21, "❌ temperatura_minimo: il fondo della U sta tra 10 e 21 °C"
print("✅ Step 7 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

`(temperatura // 3) * 3` arrotonda al multiplo di 3 inferiore: è la fascia.
`groupby("fascia_t")["mw"].mean().idxmin()` dice quale fascia ha il carico medio più basso.

</div>

<a id="sez-8"></a>
## 8. Il report

*Copilot: chat per capire. Se un errore non si capisce, "spiegami", mai "risolvilo".*

Il capo non apre notebook. Gli servono cinque frasi con i numeri e un grafico che si apre con un
doppio clic. Il grafico lo esportiamo in HTML; le frasi le scriviamo noi, con i numeri trovati
negli step precedenti.

<a id="es-8"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 8 · La sintesi**

Un file HTML e cinque frasi. Il grafico finale è lo scatter, con un titolo che dice la scoperta.

**Cosa fare.**

Ricostruisci lo scatter in `fig` con un titolo che dica dove sta il minimo ed esportalo in
`carico_nord_temperatura.html`, nella cartella del notebook. Poi stampa i numeri che
servono alle cinque frasi della cella Markdown qui sotto e completala.
Risultato atteso: un file HTML che si apre nel browser, cinque frasi con i numeri.

</div>

In [ ]:
from pathlib import Path

fig = ...
fig.write_html(...)
print(Path("carico_nord_temperatura.html").exists())

In [ ]:
# Verifica: esegui senza modificare
assert Path("carico_nord_temperatura.html").exists(), "❌ Il file HTML non c'è: write_html con il nome carico_nord_temperatura.html, nella cartella del notebook"
print("✅ Step 8 completato")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Se sei bloccato**

I numeri stanno già nelle variabili: `estremi`, `profilo`, `calo_weekend`,
`temperatura_minimo`. Una `print` con f-string per ciascuna, e la cella Markdown si compila da sola.

</div>

**Cosa abbiamo scoperto sul carico del Nord, 2024-2025**

1. I dati di Terna sono ... (non orari): ... righe pulite, tolti ... doppioni dell'ora legale di
   ottobre; le ... ore mancanti di marzo restano un buco.
2. Il giorno più pesante è stato il ... con ... MWh; il più leggero il ... con ... MWh.
3. La giornata media ha ... picchi, alle ... e alle ...; il minimo è alle ...
4. Nel weekend il carico medio cala del ...% rispetto ai giorni feriali.
5. Il carico è minimo tra ... e ... °C e cresce sia con il freddo sia con il caldo; al Nord pesa di
   più ...

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

Prova la modalità Agent su questa cella: chiedile di compilare le cinque frasi leggendo le
variabili del notebook. Poi controlla ogni numero con le `print` dello step: se uno non torna,
vince la `print`.

</div>

**Cosa ho chiesto a Copilot e cosa ho dovuto correggere**

- Ho chiesto: ...
- Ha sbagliato su: ...
- Ho verificato con: ...

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- Prima la frequenza vera e i duplicati, poi tutto il resto: un `resample` su dati sporchi mente con precisione.
- `len()` prima e dopo ogni `merge`.
- Il grafico risponde alla domanda solo se il titolo dice la risposta.

</div>

---

**Fine del notebook 12.**